# CurveNet (Runpod Edition) — Self-contained Launcher
This notebook reproduces CurveNet training on ModelNet40 while staying self-contained for Runpod: dataset cloning, conversion to CurveNet's HDF5 packs, official repo checkout, inline repo patching, training/fine-tuning, evaluation, and artifact export all happen within this notebook.


In [65]:
#@title 0) System info
import os
import platform
import subprocess
import sys
from datetime import datetime

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA in torch:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU count:', torch.cuda.device_count())
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"  - cuda:{idx} -> {props.name} ({props.total_memory/1e9:.1f} GB)")
except ImportError:
    print('PyTorch not installed; install it before running the remaining cells.')

print('Working dir:', os.getcwd())
print('Timestamp:', datetime.now())


Python: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]
Platform: Linux-6.11.0-26-generic-x86_64-with-glibc2.39
PyTorch: 2.8.0+cu128
CUDA in torch: 12.8
CUDA available: True
GPU count: 1
  - cuda:0 -> NVIDIA GeForce RTX 5090 (33.7 GB)
Working dir: /workspace/comp3419_A2b
Timestamp: 2025-11-12 08:45:01.525905


In [66]:
#@title 1) Config — paths & hyper-parameters
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

REPO_URL = 'https://github.com/tiangexiang/CurveNet.git'
REPO_BRANCH = 'main'
REPO_SUBDIR = 'CurveNet'
DATA_SUBDIR = 'modelnet40_normal_resampled'
DATASET_REPO_FOLDER = 'dataset_repo_comp3419'
DATASET_REPO_URL = 'https://github.com/Rubindai/comp3419_A2b.git'
DATASET_BRANCH = 'main'

EXP_NAME = 'curvenet_xyz_runpod'
NUM_POINTS = 1024
BATCH_SIZE = 32
TEST_BATCH_SIZE = 16
EPOCHS = 200
FINE_TUNE_EPOCHS = 280
LR = 0.001
FINE_TUNE_LR = 3e-4
SCHEDULER = 'cos'
USE_SGD = True
NUM_WORKERS = 12
PIN_MEMORY = True
AUTO_FETCH_DATASET = True
INSTALL_REQUIREMENTS = True
APPLY_RUNPOD_PATCH = True

REPO_DIR = (NOTEBOOK_DIR / REPO_SUBDIR).resolve()
REPO_MARKER = REPO_DIR / '.prepared_from_git'
CORE_DIR = REPO_DIR / 'core'
DATA_PATH = (NOTEBOOK_DIR / DATA_SUBDIR).resolve()
DATASET_WORKSPACE = (NOTEBOOK_DIR / DATASET_REPO_FOLDER).resolve()
TARGET_DATA_DIR = CORE_DIR / 'data' / 'modelnet40_ply_hdf5_2048'
LOG_DIR = CORE_DIR / 'log' / EXP_NAME

LOG_DIR.mkdir(parents=True, exist_ok=True)
print('Notebook dir :', NOTEBOOK_DIR)
print('Repo dir     :', REPO_DIR)
print('Dataset path :', DATA_PATH)
print('Target H5   :', TARGET_DATA_DIR)
print('Log dir      :', LOG_DIR)


Notebook dir : /workspace/comp3419_A2b
Repo dir     : /workspace/comp3419_A2b/CurveNet
Dataset path : /workspace/comp3419_A2b/modelnet40_normal_resampled
Target H5   : /workspace/comp3419_A2b/CurveNet/core/data/modelnet40_ply_hdf5_2048
Log dir      : /workspace/comp3419_A2b/CurveNet/core/log/curvenet_xyz_runpod


In [67]:
#@title 2) Clone assignment repo + sync ModelNet40
import shutil
import subprocess
import sys

if AUTO_FETCH_DATASET and not DATA_PATH.exists():
    workspace = DATASET_WORKSPACE
    if workspace.exists():
        print('Dataset repo already exists at', workspace, '- skipping fetch/reset (delete folder to refresh).')
    else:
        workspace.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', DATASET_BRANCH,
            DATASET_REPO_URL, str(workspace)
        ], check=True)
    src = workspace / DATA_SUBDIR
    if not src.exists():
        raise FileNotFoundError(f'Missing {src} inside dataset repo')
    shutil.copytree(src, DATA_PATH, dirs_exist_ok=True)
    print('Copied dataset to', DATA_PATH)
else:
    if DATA_PATH.exists():
        print('Dataset already exists at', DATA_PATH)
    else:
        print('AUTO_FETCH_DATASET disabled; please populate', DATA_PATH)


Dataset already exists at /workspace/comp3419_A2b/modelnet40_normal_resampled


In [68]:
#@title 3) Clone/Pull official CurveNet repo + install deps
import subprocess
import sys
import shutil
import time

repo_git = REPO_DIR / '.git'
repo_marker = REPO_MARKER

def stamp_repo(message):
    stamp_line = f"{message} @ {time.ctime()}"
    repo_marker.write_text(stamp_line + "\n")
def ensure_repo_materialized():
    if REPO_DIR.exists():
        if repo_git.exists():
            print('CurveNet repo with git metadata already present at', REPO_DIR)
            if not repo_marker.exists():
                stamp_repo('existing git checkout')
            return False
        if repo_marker.exists():
            print('CurveNet source already prepared at', REPO_DIR)
            return False
        print('CurveNet directory exists without git metadata; marking it as prepared (manual copy).')
        stamp_repo('manual copy')
        return False
    print('Cloning CurveNet fresh into', REPO_DIR)
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    if repo_git.exists():
        shutil.rmtree(repo_git)
    stamp_repo('cloned from upstream')
    return True

ensure_repo_materialized()
if INSTALL_REQUIREMENTS:
    requirements = REPO_DIR / 'requirements.txt'
    if requirements.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=False)
    extras = ['h5py', 'scikit-learn', 'tqdm', 'matplotlib', 'numpy']
    subprocess.run([sys.executable, '-m', 'pip', 'install', *extras], check=False)
print('Repo ready at', REPO_DIR)


Repo already exists at /workspace/comp3419_A2b/CurveNet - assuming it is current.
Repo ready at /workspace/comp3419_A2b/CurveNet


In [69]:
#@title 3) Build CurveNet HDF5 dataset + patch core/data.py
import glob
import h5py
import numpy as np
import shutil
from pathlib import Path
import re
import subprocess
import time

repo_git = REPO_DIR / '.git'
repo_marker = REPO_MARKER
if not REPO_DIR.exists():
    print('CurveNet repo missing while building HDF5, cloning now so CORE_DIR is available.')
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    if repo_git.exists():
        shutil.rmtree(repo_git)
    repo_marker.write_text(f'autoclone for dataset @ {time.ctime()}\n')
elif repo_git.exists() and not repo_marker.exists():
    print('CurveNet repo has git metadata; leaving it as-is.')
    repo_marker.write_text(f'manual git checkout @ {time.ctime()}\n')
elif not repo_marker.exists():
    print('CurveNet directory present without marker; marking as prepared.')
    repo_marker.write_text(f'manual copy detected @ {time.ctime()}\n')
else:
    print('CurveNet source already prepared at', REPO_DIR)

TARGET_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
shape_file = DATA_PATH / 'modelnet40_shape_names.txt'
if not shape_file.exists():
    raise FileNotFoundError(f'Missing {shape_file}')
classes = [line.strip() for line in shape_file.read_text().splitlines() if line.strip()]
class_to_id = {name: idx for idx, name in enumerate(classes)}
shutil.copy2(shape_file, TARGET_DATA_DIR.parent / 'modelnet40_shape_names.txt')

for partition in ('train', 'test'):
    h5_path = TARGET_DATA_DIR / f'ply_data_{partition}.h5'
    entries = [
        line.strip()
        for line in (DATA_PATH / f'modelnet40_{partition}.txt').read_text().splitlines()
        if line.strip()
    ]
    data = []
    labels = []
    rng = np.random.default_rng(42)
    for entry in entries:
        category = entry.rsplit('_', 1)[0]
        file_path = DATA_PATH / category / f"{entry}.txt"
        if not file_path.exists():
            raise FileNotFoundError(f'Missing point file {file_path}')
        pts = np.loadtxt(file_path, delimiter=',', dtype=np.float32)[:, :3]
        n, _ = pts.shape
        if n >= NUM_POINTS:
            idx = rng.choice(n, NUM_POINTS, replace=False)
        else:
            idx = rng.choice(n, NUM_POINTS, replace=True)
        sampled = pts[idx]
        data.append(sampled)
        labels.append(class_to_id[category])
    data = np.stack(data).astype('float32')
    labels = np.array(labels, dtype='int64').reshape(-1, 1)
    TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with h5py.File(h5_path, 'w') as f:
        f.create_dataset('data', data=data)
        f.create_dataset('label', data=labels)
    print('Wrote', h5_path, 'with', data.shape[0], 'examples')
    listing = TARGET_DATA_DIR / f'{partition}_files.txt'
    listing.write_text(str(h5_path) + '\n')

# Patch core/data.py to use absolute dataset root
core_data = CORE_DIR / 'data.py'
content = core_data.read_text()
abs_dir = str((CORE_DIR / 'data').resolve()) + '/'
new = re.sub(r"^DATA_DIR\s*=\s*['\"]([^'\"]+)['\"]", f"DATA_DIR = '{abs_dir}'", content, flags=re.M)
if new == content:
    print('core/data.py already pointed to', abs_dir)
else:
    core_data.write_text(new)
    print('Patched core/data.py ->', abs_dir)


CurveNet repo already present at /workspace/comp3419_A2b/CurveNet
Wrote /workspace/comp3419_A2b/CurveNet/core/data/modelnet40_ply_hdf5_2048/ply_data_train.h5 with 9843 examples
Wrote /workspace/comp3419_A2b/CurveNet/core/data/modelnet40_ply_hdf5_2048/ply_data_test.h5 with 2468 examples
Patched core/data.py -> /workspace/comp3419_A2b/CurveNet/core/data/


In [70]:
#@title 5) Patch CurveNet scripts for Runpod
from pathlib import Path

main_cls = CORE_DIR / 'main_cls.py'
text = main_cls.read_text()

snippet = (
    "    parser.add_argument('--num_workers', type=int, default=8, help='workers for dataloaders')\n"
    "    parser.add_argument('--pin_memory', action='store_true', default=False, help='pin memory flag')\n"
)
pattern = (
    "    parser.add_argument('--test_batch_size', type=int, default=16, metavar='batch_size',\n"
    "                        help='Size of batch)')\n"
)
if '--pin_memory' not in text:
    if pattern not in text:
        raise RuntimeError('Could not find insertion point for parser args')
    text = text.replace(pattern, pattern + snippet, 1)

text = text.replace('num_workers=8,', 'num_workers=args.num_workers, pin_memory=args.pin_memory,', 2)
text = text.replace('torch.load(args.model_path)', 'torch.load(args.model_path, weights_only=False)')
if text != main_cls.read_text():
    main_cls.write_text(text)
    print('Patched main_cls.py for Runpod.')
else:
    print('main_cls.py already patched.')


Patched main_cls.py for Runpod.


In [71]:

#@title 6) Validate dataset and patch outcome
import os

if not TARGET_DATA_DIR.exists():
    raise FileNotFoundError('CurveNet HDF5 data missing at ' + str(TARGET_DATA_DIR))
print('HDF5 dataset ready at', TARGET_DATA_DIR)
checkpoint_dir = REPO_DIR / 'checkpoints' / EXP_NAME / 'models'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print('Checkpoints dir ensured at', checkpoint_dir)


HDF5 dataset ready at /workspace/comp3419_A2b/CurveNet/core/data/modelnet40_ply_hdf5_2048
Checkpoints dir ensured at /workspace/comp3419_A2b/CurveNet/checkpoints/curvenet_xyz_runpod/models


In [72]:

#@title 7) Helper functions for training/testing
import os
import shlex
import subprocess
import sys
from pathlib import Path

if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))

LOG_PATH = CORE_DIR / 'log'
LOG_PATH.mkdir(exist_ok=True)
BEST_MODEL = CORE_DIR / 'checkpoints' / EXP_NAME / 'models' / 'model.t7'


def stream_cmd(cmd, log_file):
    print('Running command:', ' '.join(shlex.quote(str(c)) for c in cmd))
    with open(log_file, 'a') as handle:
        proc = subprocess.Popen(cmd, cwd=CORE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
            handle.write(line)
        proc.wait()
        if proc.returncode != 0:
            raise RuntimeError(f'Command failed with exit code {proc.returncode}')


def train_cmd(epochs, lr):
    cmd = [
        sys.executable,
        'main_cls.py',
        '--exp_name', EXP_NAME,
        '--num_points', str(NUM_POINTS),
        '--batch_size', str(BATCH_SIZE),
        '--test_batch_size', str(TEST_BATCH_SIZE),
        '--epochs', str(epochs),
        '--lr', str(lr),
        '--scheduler', SCHEDULER,
        '--num_workers', str(NUM_WORKERS),
    ]
    if PIN_MEMORY:
        cmd.append('--pin_memory')
    log_file = LOG_PATH / f"{EXP_NAME}_console_train.txt"
    stream_cmd(cmd, log_file)
    return log_file


def test_cmd(model_path):
    cmd = [
        sys.executable,
        'main_cls.py',
        '--exp_name', EXP_NAME,
        '--num_points', str(NUM_POINTS),
        '--test_batch_size', str(TEST_BATCH_SIZE),
        '--eval', 'True',
        '--model_path', str(model_path),
        '--num_workers', str(NUM_WORKERS),
    ]
    if PIN_MEMORY:
        cmd.append('--pin_memory')
    log_file = LOG_PATH / f"{EXP_NAME}_console_eval.txt"
    stream_cmd(cmd, log_file)
    return log_file


In [73]:
#@title 8) Train baseline (SSG-inspired) + log
train_cmd(EPOCHS, LR)


Running command: /usr/local/bin/python main_cls.py --exp_name curvenet_xyz_runpod --num_points 1024 --batch_size 32 --test_batch_size 16 --epochs 200 --lr 0.001 --scheduler cos --num_workers 12 --pin_memory
Namespace(exp_name='curvenet_xyz_runpod', dataset='modelnet40', batch_size=32, test_batch_size=16, num_workers=12, pin_memory=True, epochs=200, use_sgd=True, lr=0.001, momentum=0.9, scheduler='cos', no_cuda=False, eval=False, num_points=1024, model_path='')
random seed is: 7637
Using GPU : 0 from 1 devices
Let's use1GPUs!
Use SGD
Train 0, loss: 2.578223, train acc: 0.510281
Test 0, loss: 2.180635, test acc: 0.630875
best: 0.631
Train 1, loss: 2.179209, train acc: 0.672638
Test 1, loss: 1.911404, test acc: 0.762561
best: 0.763
Train 2, loss: 2.029770, train acc: 0.713253
Test 2, loss: 1.854198, test acc: 0.775527
best: 0.776
Train 3, loss: 1.938394, train acc: 0.753054
Test 3, loss: 1.808342, test acc: 0.809968
best: 0.810
Train 4, loss: 1.885301, train acc: 0.774939
Test 4, loss: 1.

PosixPath('/workspace/comp3419_A2b/CurveNet/core/log/curvenet_xyz_runpod_console_train.txt')

In [74]:
#@title 9) Fine-tune baseline
train_cmd(FINE_TUNE_EPOCHS, FINE_TUNE_LR)


Running command: /usr/local/bin/python main_cls.py --exp_name curvenet_xyz_runpod --num_points 1024 --batch_size 32 --test_batch_size 16 --epochs 280 --lr 0.0003 --scheduler cos --num_workers 12 --pin_memory
Namespace(exp_name='curvenet_xyz_runpod', dataset='modelnet40', batch_size=32, test_batch_size=16, num_workers=12, pin_memory=True, epochs=280, use_sgd=True, lr=0.0003, momentum=0.9, scheduler='cos', no_cuda=False, eval=False, num_points=1024, model_path='')
random seed is: 9309
Using GPU : 0 from 1 devices
Let's use1GPUs!
Use SGD
Train 0, loss: 2.449636, train acc: 0.541836
Test 0, loss: 1.985302, test acc: 0.725284
best: 0.725
Train 1, loss: 2.046487, train acc: 0.697374
Test 1, loss: 1.831318, test acc: 0.804700
best: 0.805
Train 2, loss: 1.936297, train acc: 0.750713
Test 2, loss: 1.836059, test acc: 0.794165
best: 0.805
Train 3, loss: 1.877447, train acc: 0.779316
Test 3, loss: 1.775267, test acc: 0.809562
best: 0.810
Train 4, loss: 1.833461, train acc: 0.795297
Test 4, loss: 

PosixPath('/workspace/comp3419_A2b/CurveNet/core/log/curvenet_xyz_runpod_console_train.txt')

In [ ]:
#@title 10) Evaluate checkpoint only (no extra training)
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'Best model not found at {BEST_MODEL}')
test_cmd(BEST_MODEL)


In [ ]:
#@title 11) Heads/Tails — quick excerpts for your report
from pathlib import Path

def tail(path, n=20):
    if not path.exists():
        return []
    with open(path) as f:
        lines = f.readlines()
    return lines[-n:]

train_log = CORE_DIR / 'log' / f"{EXP_NAME}_console_train.txt"
eval_log = CORE_DIR / 'log' / f"{EXP_NAME}_console_eval.txt"
print(f"=== Train log (last lines) ===\n{''.join(tail(train_log))}")
print(f"=== Eval log (last lines) ===\n{''.join(tail(eval_log))}")


In [ ]:
#@title 12) Plot accuracy curves from logs
import matplotlib.pyplot as plt
import re

train_log = CORE_DIR / 'log' / f"{EXP_NAME}_console_train.txt"
train_acc = []
test_acc = []
if train_log.exists():
    for line in train_log.read_text().splitlines():
        if 'test acc' in line.lower() and 'best' in line.lower():
            continue
        m = re.search(r'test acc: ([0-9.]+)', line)
        if m:
            test_acc.append(float(m.group(1)))
        m2 = re.search(r'train acc: ([0-9.]+)', line)
        if m2:
            train_acc.append(float(m2.group(1)))
if train_acc or test_acc:
    epochs = list(range(1, len(train_acc) + 1))
    plt.figure(figsize=(6, 4))
    if train_acc:
        plt.plot(epochs[:len(train_acc)], train_acc, label='Train acc')
    if test_acc:
        plt.plot(epochs[:len(test_acc)], test_acc, label='Test acc')
    plt.legend()
    plt.xlabel('Epoch (per logged step)')
    plt.ylabel('Accuracy')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No accuracy logs yet. Run training cells first.')


In [ ]:
#@title 13) Confusion matrix & per-class accuracy snapshot
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import confusion_matrix

model = torch.nn.DataParallel(__import__('models.curvenet_cls').models.curvenet_cls.CurveNet())
state = torch.load(BEST_MODEL, map_location='cpu')
model.load_state_dict(state)
model.eval()

from data import ModelNet40
loader = torch.utils.data.DataLoader(ModelNet40(num_points=NUM_POINTS, partition='test'),
                                     batch_size=TEST_BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
true, pred = [], []
with torch.no_grad():
    for pts, labels in loader:
        pts = pts.permute(0, 2, 1)
        outputs = model(pts)
        y = outputs.max(1)[1]
        true.append(labels.numpy())
        pred.append(y.numpy())
true = np.concatenate(true)
pred = np.concatenate(pred)
cm = confusion_matrix(true, pred)
analysis = CORE_DIR / 'analysis'
analysis.mkdir(exist_ok=True)
np.savetxt(analysis / 'confusion_matrix.csv', cm, fmt='%d', delimiter=',')
per_class = cm.diagonal() / cm.sum(axis=1).clip(min=1)
np.savetxt(analysis / 'per_class_accuracy.csv', per_class, fmt='%.6f', delimiter=',')
plt.figure(figsize=(6, 6))
plt.imshow(cm, cmap='Blues')
plt.title('CurveNet confusion matrix')
plt.colorbar()
plt.tight_layout()
plt.savefig(analysis / 'confusion_matrix.png', dpi=180)
plt.show()
print('Saved confusion matrix to', analysis)


In [ ]:
#@title 14) Export artifacts bundle
import shutil
import subprocess
import time
from pathlib import Path

BASE = Path('curvenet_artifacts')
TARGET = BASE / EXP_NAME
if TARGET.exists():
    shutil.rmtree(TARGET)
shutil.copytree(CORE_DIR / 'log', TARGET / 'logs')
shutil.copytree(CORE_DIR / 'checkpoints', TARGET / 'checkpoints')
note = TARGET / 'NOTE.txt'
note.write_text('\n'.join([
    f'Exported at {time.ctime()}',
    f'Notebook: {NOTEBOOK_DIR}',
    'Repo commit: ' + subprocess.getoutput(f"cd '{REPO_DIR}' && git rev-parse HEAD"),
]))
print('Exported to', TARGET)
